In [1]:
from google.colab import drive
import pandas as pd
import json 
import torch

drive.mount('/content/drive')

SAVE_PATH = '/content/drive/MyDrive/sentiment_project/'

with open(SAVE_PATH + 'vocab.json', 'r') as f:
    word2idx = json.load(f)

train_df = pd.read_csv(SAVE_PATH + 'train_preprocessed.csv')
val_df   = pd.read_csv(SAVE_PATH + 'val_preprocessed.csv')
test_df  = pd.read_csv(SAVE_PATH + 'test_preprocessed.csv')

Mounted at /content/drive


In [2]:
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from collections import Counter

class TweetDataset(Dataset):
    def __init__(self, texts, labels, word2idx, max_length=50):
        self.texts      = texts
        self.labels     = labels
        self.word2idx   = word2idx
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        tokens  = self.texts[idx].split()[:self.max_length]
        tokens  = ['<START>'] + tokens + ['<END>']
        indices = [self.word2idx.get(t, self.word2idx['<UNK>']) for t in tokens]
        return {
            'text':    self.texts[idx],
            'indices': torch.tensor(indices, dtype=torch.long),
            'label':   torch.tensor(self.labels[idx], dtype=torch.long),
            'length':  len(indices),
        }

def collate_fn(batch):
    return {
        'texts':   [item['text'] for item in batch],
        'indices': pad_sequence(
            [item['indices'] for item in batch],
            batch_first=True, padding_value=0
        ),
        'labels':  torch.stack([item['label']  for item in batch]),
        'lengths': torch.tensor([item['length'] for item in batch]),
    }

# Build datasets
train_dataset = TweetDataset(train_df['preprocessed_text'].tolist(), train_df['label'].tolist(), word2idx)
val_dataset   = TweetDataset(val_df['preprocessed_text'].tolist(),   val_df['label'].tolist(),   word2idx)
test_dataset  = TweetDataset(test_df['preprocessed_text'].tolist(),  test_df['label'].tolist(),  word2idx)

# Build dataloaders
SEED = 42
g = torch.Generator()
g.manual_seed(SEED)
BATCH_SIZE = 32

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  collate_fn=collate_fn, generator=g)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches:   {len(val_loader)}")
print(f"Test batches:  {len(test_loader)}")

Train batches: 1426
Val batches:   63
Test batches:  384


In [3]:
import torch 
import numpy as np


torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

Using device: cuda


In [4]:
import torch.nn as nn

class sentimentGRU(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim, n_layers, dropout ):
        super().__init__()
        
        self.embedding = nn.Embedding(vocab_size , embedding_dim)
        
        self.gru = nn.GRU(
            embedding_dim,
            hidden_dim,
            num_layers=n_layers,
            bidirectional=False,
            dropout=dropout if n_layers > 1 else 0,
            batch_first=True,
        )
        
        self.fc = nn.Linear(hidden_dim , output_dim)
        
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x):
        x = self.embedding(x)
        output, h = self.gru(x)
        x = self.dropout(h[-1])
        x = self.fc(x)
        return x

In [5]:
class sentimentLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim, n_layers, dropout):
        super().__init__()
        
        self.embedding = nn.Embedding(vocab_size , embedding_dim)
        
        self.lstm = nn.LSTM(
            embedding_dim,
            hidden_dim,
            num_layers=n_layers,
            bidirectional=False,
            dropout=dropout if n_layers > 1 else 0,
            batch_first=True,
        )
        
        self.fc = nn.Linear(hidden_dim,output_dim)
        
        self.dropout = nn.Dropout(dropout)
        
    def forward(self , x):
        x = self.embedding(x)
        output , (h , c ) = self.lstm(x)
        x = self.dropout(h[-1])
        x = self.fc(x)
        return x
     

In [6]:
import math

class PositionalEncoding(nn.Module):
    def __init__(self, embedding_dim, max_len=512, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        pe = torch.zeros(max_len, embedding_dim)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(
            torch.arange(0, embedding_dim, 2).float() *
            (-math.log(10000.0) / embedding_dim)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))  

    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)


class SentimentTransformer(nn.Module):
    def __init__(self, vocab_size, embedding_dim, num_classes,
                 num_heads=4, num_layers=2, ff_dim=256, dropout=0.3):
        super().__init__()
        self.embedding     = nn.Embedding(vocab_size, embedding_dim)
        self.pos_encoding  = PositionalEncoding(embedding_dim, dropout=dropout)

        encoder_layer      = nn.TransformerEncoderLayer(
            d_model=embedding_dim,
            nhead=num_heads,
            dim_feedforward=ff_dim,
            dropout=dropout,
            batch_first=True      
        )
        self.transformer   = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.dropout       = nn.Dropout(dropout)
        self.fc            = nn.Linear(embedding_dim, num_classes)

    def forward(self, x):
        x = self.pos_encoding(self.embedding(x)) 
        x = self.transformer(x)                   
        x = x.mean(dim=1)                          
        x = self.fc(self.dropout(x))

        return x

In [10]:
import torch.optim as optim
from sklearn.metrics import accuracy_score

# ── Hyperparameters ──────────────────────────────
EMBEDDING_DIM = 128
HIDDEN_DIM    = 128
LEARNING_RATE = 0.001
EPOCHS        = 10
DROPOUT       = 0.5
NUM_CLASSES   = 3
PATIENCE      = 3
vocab_size    = len(word2idx)

# ── Model definitions ────────────────────────────
modelGru = sentimentGRU(
    vocab_size, EMBEDDING_DIM, HIDDEN_DIM,
    output_dim=NUM_CLASSES, n_layers=2, dropout=DROPOUT
).to(DEVICE)

modelLstm = sentimentLSTM(
    vocab_size, EMBEDDING_DIM, HIDDEN_DIM,
    output_dim=NUM_CLASSES, n_layers=2, dropout=DROPOUT
).to(DEVICE)

modelTransformer = SentimentTransformer(
    vocab_size, EMBEDDING_DIM,
    num_classes=NUM_CLASSES, dropout=DROPOUT
).to(DEVICE)

criterion = nn.CrossEntropyLoss()

def train(model, train_loader, val_loader, optimizer, criterion, epochs, device, patience=PATIENCE):
    best_val_acc = 0.0
    patience_counter = 0
    
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', patience=2, factor=0.5
    )
    
    for epoch in range(epochs):

        # ── Phase 1: Training ──
        model.train()
        running_loss = 0.0
        correct      = 0
        total        = 0

        for batch in train_loader:
            inputs  = batch['indices'].to(device)
            targets = batch['labels'].to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss    = criterion(outputs, targets)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            running_loss += loss.item()
            predicted     = outputs.argmax(dim=1)
            correct      += (predicted == targets).sum().item()
            total        += targets.size(0)

        train_loss = running_loss / len(train_loader)
        train_acc  = correct / total

        # ── Phase 2: Validation ──
        model.eval()
        val_correct = 0
        val_total   = 0

        with torch.no_grad():
            for batch in val_loader:
                inputs  = batch['indices'].to(device)
                targets = batch['labels'].to(device)

                outputs   = model(inputs)
                predicted = outputs.argmax(dim=1)

                val_correct += (predicted == targets).sum().item()
                val_total   += targets.size(0)

        val_acc = val_correct / val_total
        
        scheduler.step(val_acc)

        print(f"Epoch {epoch+1}/{epochs} | "
              f"Train Loss: {train_loss:.4f} | "
              f"Train Acc: {train_acc*100:.2f}% | "
              f"Val Acc: {val_acc*100:.2f}%")

        # ── Save best model ──
        if val_acc > best_val_acc:
            best_val_acc     = val_acc
            patience_counter = 0   # ← reset counter when improving
            torch.save(model.state_dict(), SAVE_PATH + f'{model.__class__.__name__}_best.pt')
            print(f"   Best model saved, Val Acc: {val_acc*100:.2f}%")
        else:
            patience_counter += 1  # ← count epochs without improvement
            print(f"  No improvement ({patience_counter}/{patience})")

            if patience_counter >= patience:   # ← stop if no improvement
                print(f"  Early stopping at epoch {epoch+1}")
                break

    return best_val_acc


# ── Train each model ─────────────────────────────
print("=" * 40)
print("Training GRU")
print("=" * 40)
optimizer = optim.Adam(modelGru.parameters(), lr=LEARNING_RATE, weight_decay=1e-3)
train(modelGru, train_loader, val_loader, optimizer, criterion, EPOCHS, DEVICE)
torch.save(modelGru.state_dict(), '/content/drive/MyDrive/sentiment_project/gru_best.pt')

print("=" * 40)
print("Training LSTM")
print("=" * 40)
optimizer = optim.Adam(modelLstm.parameters(), lr=LEARNING_RATE, weight_decay=1e-3)
train(modelLstm, train_loader, val_loader, optimizer, criterion, EPOCHS, DEVICE)
torch.save(modelLstm.state_dict(), '/content/drive/MyDrive/sentiment_project/lstm_best.pt')

print("=" * 40)
print("Training Transformer")
print("=" * 40)
optimizer = optim.Adam(modelTransformer.parameters(), lr=LEARNING_RATE, weight_decay=1e-3)
train(modelTransformer, train_loader, val_loader, optimizer, criterion, EPOCHS, DEVICE)
torch.save(modelTransformer.state_dict(), '/content/drive/MyDrive/sentiment_project/transformer_best.pt')

Training GRU
Epoch 1/10 | Train Loss: 0.9601 | Train Acc: 54.71% | Val Acc: 57.60%
   Best model saved, Val Acc: 57.60%
Epoch 2/10 | Train Loss: 0.8827 | Train Acc: 60.60% | Val Acc: 62.55%
   Best model saved, Val Acc: 62.55%
Epoch 3/10 | Train Loss: 0.8163 | Train Acc: 63.29% | Val Acc: 65.35%
   Best model saved, Val Acc: 65.35%
Epoch 4/10 | Train Loss: 0.7887 | Train Acc: 64.98% | Val Acc: 65.65%
   Best model saved, Val Acc: 65.65%
Epoch 5/10 | Train Loss: 0.7812 | Train Acc: 65.24% | Val Acc: 63.25%
  No improvement (1/3)
Epoch 6/10 | Train Loss: 0.7817 | Train Acc: 65.05% | Val Acc: 65.15%
  No improvement (2/3)
Epoch 7/10 | Train Loss: 0.7754 | Train Acc: 65.47% | Val Acc: 66.25%
   Best model saved, Val Acc: 66.25%
Epoch 8/10 | Train Loss: 0.7713 | Train Acc: 65.82% | Val Acc: 67.55%
   Best model saved, Val Acc: 67.55%
Epoch 9/10 | Train Loss: 0.7734 | Train Acc: 65.89% | Val Acc: 66.15%
  No improvement (1/3)
Epoch 10/10 | Train Loss: 0.7708 | Train Acc: 66.02% | Val Acc: 65

so i got in first try an **overfitting** in GRU and the two others need more learn so i make epoch = 10
i think that will make it up to `+70%` i think . 

the second try i got overfitting on all the models just after the 5th epoch . still try to know why :/

the third try is bad . like very bad . i try to chnage parameters and change the train function but still no improvment 
**so i think i will stop know** i need to learn more about how to buuild a model and create a train function . 
-- for now im proud of what i did , but i need more i will stop here for now . 